In [0]:
GOLD_PATH = "/Volumes/workspace/default/logs_volume/gold"
SILVER_PATH = "/Volumes/workspace/default/logs_volume/silver"

In [0]:
silver_df = spark.read.format("delta").load(SILVER_PATH)

print(f"Rows loaded: {silver_df.count():,}")

Rows loaded: 1,886,836


In [0]:
from pyspark.sql.functions import (
    col, count, sum, avg, max, min, when,
    hour, date_trunc, round,
    window, lag, rank
)

hourly_df = (silver_df
    .withColumn("hour_bucket", date_trunc("hour", col("timestamp")))
    .groupBy("hour_bucket")
    .agg(
        count("*").alias("total_requests"),
        sum("bytes").alias("total_bytes"),
        count(when(col("status") == 404, 1)).alias("errors_404"),
        count(when(col("status") == 500, 1)).alias("errors_500"),
        count(when(col("status").between(200, 299), 1)).alias("success_count")
    )
    .withColumn("error_rate",
        round((col("errors_404") + col("errors_500")) / col("total_requests"), 4)
    )
    .withColumn("success_rate",
        round(col("success_count") / col("total_requests"), 4)
    )
    .orderBy("hour_bucket")
)

hourly_df.show(10)

+-------------------+--------------+-----------+----------+----------+-------------+----------+------------+
|        hour_bucket|total_requests|total_bytes|errors_404|errors_500|success_count|error_rate|success_rate|
+-------------------+--------------+-----------+----------+----------+-------------+----------+------------+
|1995-07-01 04:00:00|          3558|   80024069|        24|         0|         3185|    0.0067|      0.8952|
|1995-07-01 05:00:00|          2989|   81415526|        10|         0|         2708|    0.0033|       0.906|
|1995-07-01 06:00:00|          2268|   57160621|        12|         0|         2040|    0.0053|      0.8995|
|1995-07-01 07:00:00|          1721|   36964257|        15|         0|         1551|    0.0087|      0.9012|
|1995-07-01 08:00:00|          1474|   29295642|        10|         0|         1341|    0.0068|      0.9098|
|1995-07-01 09:00:00|          1340|   32893213|         9|         0|         1165|    0.0067|      0.8694|
|1995-07-01 10:00:0

In [0]:
from pyspark.sql.window import Window
# Define the window: partition by nothing (whole dataset), order by time, 
# look back 2 rows (= current + 2 previous hours = 3hr window)
hourly_window = (Window
    .orderBy("hour_bucket")
    .rowsBetween(-2, 0))  # -2 = 2 rows back, 0 = current row

hourly_df = hourly_df.withColumn(
    "rolling_3hr_avg_requests",
    round(avg("total_requests").over(hourly_window), 2)
)

hourly_df.show(10)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+--------------+-----------+----------+----------+-------------+----------+------------+------------------------+
|        hour_bucket|total_requests|total_bytes|errors_404|errors_500|success_count|error_rate|success_rate|rolling_3hr_avg_requests|
+-------------------+--------------+-----------+----------+----------+-------------+----------+------------+------------------------+
|1995-07-01 04:00:00|          3558|   80024069|        24|         0|         3185|    0.0067|      0.8952|                  3558.0|
|1995-07-01 05:00:00|          2989|   81415526|        10|         0|         2708|    0.0033|       0.906|                  3273.5|
|1995-07-01 06:00:00|          2268|   57160621|        12|         0|         2040|    0.0053|      0.8995|                 2938.33|
|1995-07-01 07:00:00|          1721|   36964257|        15|         0|         1551|    0.0087|      0.9012|                  2326.0|
|1995-07-01 08:00:00|          1474|   29295642|        10|   

In [0]:
from pyspark.sql.functions import countDistinct
ip_df = (silver_df
    .groupBy("host")
    .agg(
        count("*").alias("total_requests"),
        sum("bytes").alias("total_bytes_transferred"),
        count(when(col("status") >= 400, 1)).alias("total_errors"),
        round(
            count(when(col("status") >= 400, 1)) / count("*") * 100, 2
        ).alias("error_rate_pct"),
        min("timestamp").alias("first_seen"),
        max("timestamp").alias("last_seen"),
        countDistinct("endpoint").alias("unique_endpoints_hit")
    )
    .orderBy("total_requests", ascending=False)
)

ip_df.show(10)

+--------------------+--------------+-----------------------+------------+--------------+-------------------+-------------------+--------------------+
|                host|total_requests|total_bytes_transferred|total_errors|error_rate_pct|         first_seen|          last_seen|unique_endpoints_hit|
+--------------------+--------------+-----------------------+------------+--------------+-------------------+-------------------+--------------------+
|piweba3y.prodigy.com|         17572|              433605604|         110|          0.63|1995-07-01 04:00:54|1995-07-28 17:01:44|                1705|
|piweba4y.prodigy.com|         11591|              250619888|          56|          0.48|1995-07-01 06:37:33|1995-07-28 17:27:38|                1210|
|piweba1y.prodigy.com|          9868|              261097586|          92|          0.93|1995-07-01 04:11:33|1995-07-28 17:19:09|                1282|
|  alyssa.prodigy.com|          7852|              209657138|          54|          0.69|1995-

In [0]:
ip_hourly_df = (silver_df
    .groupBy(
        "host",
        window(col("timestamp"), "1 hour").alias("time_window")
    )
    .agg(count("*").alias("requests_in_hour"))
    .select(
        col("host"),
        col("time_window.start").alias("hour_start"),
        col("requests_in_hour")
    )
    .orderBy("host", "hour_start")
)

ip_hourly_df.show(10)

+--------------------+-------------------+----------------+
|                host|         hour_start|requests_in_hour|
+--------------------+-------------------+----------------+
|         ***.novo.dk|1995-07-11 12:00:00|              16|
|      007.thegap.com|1995-07-06 21:00:00|              15|
|      007.thegap.com|1995-07-06 23:00:00|              15|
|      007.thegap.com|1995-07-10 00:00:00|               2|
|      007.thegap.com|1995-07-10 01:00:00|               9|
|      007.thegap.com|1995-07-23 20:00:00|               4|
|01-dynamic-c.rott...|1995-07-28 16:00:00|               1|
|01-dynamic-c.woki...|1995-07-05 11:00:00|               2|
|01-dynamic-c.woki...|1995-07-05 12:00:00|               1|
|01-dynamic-c.woki...|1995-07-10 12:00:00|              11|
+--------------------+-------------------+----------------+
only showing top 10 rows


In [0]:
endpoint_df = (silver_df
    .groupBy("endpoint")
    .agg(
        count("*").alias("total_hits"),
        round(avg("bytes"), 2).alias("avg_bytes"),
        max("bytes").alias("max_bytes"),
        count(when(col("status") == 404, 1)).alias("not_found_count"),
        count(when(col("status") == 200, 1)).alias("success_count"),
        countDistinct("host").alias("unique_visitors")
    )
    .orderBy("total_hits", ascending=False)
)

endpoint_df.show(20)

+--------------------+----------+---------+---------+---------------+-------------+---------------+
|            endpoint|total_hits|avg_bytes|max_bytes|not_found_count|success_count|unique_visitors|
+--------------------+----------+---------+---------+---------------+-------------+---------------+
|/images/NASA-logo...|    111086|   633.56|      786|              0|        90076|          49583|
|/images/KSC-logos...|     89529|  1032.87|     1204|              0|        77094|          49049|
|/images/MOSAIC-lo...|     60299|   320.71|      363|              0|        53671|          29729|
|/images/USA-logos...|     59844|   206.69|      234|              0|        53267|          29490|
|/images/WORLD-log...|     59324|    592.1|      669|              0|        52911|          29244|
|/images/ksclogo-m...|     58615|  5256.13|     5866|              0|        52778|          27811|
|/images/launch-lo...|     40841|  1514.11|     1713|              0|        36213|          23242|


In [0]:
broken_endpoints = (endpoint_df
    .withColumn(
        "not_found_rate_pct",
        round(col("not_found_count") / col("total_hits") * 100, 2)
    )
    .filter(col("total_hits") > 50)   # ignore low-traffic endpoints
    .orderBy("not_found_rate_pct", ascending=False)
)

broken_endpoints.show(20)

+--------------------+----------+---------+---------+---------------+-------------+---------------+------------------+
|            endpoint|total_hits|avg_bytes|max_bytes|not_found_count|success_count|unique_visitors|not_found_rate_pct|
+--------------------+----------+---------+---------+---------------+-------------+---------------+------------------+
|/://spacelink.msf...|       215|      0.0|        0|            215|            0|            178|             100.0|
|/pub/winvn/readme...|       667|      0.0|        0|            667|            0|            548|             100.0|
|/history/apollo/p...|       215|      0.0|        0|            215|            0|             72|             100.0|
|/history/apollo/s...|       183|      0.0|        0|            183|            0|             52|             100.0|
|/history/apollo/s...|        60|      0.0|        0|             60|            0|             20|             100.0|
|/shuttle/resource...|       180|      0.0|     

In [0]:
# Hourly stats
(hourly_df
    .write.format("delta")
    .mode("overwrite")
    .save(f"{GOLD_PATH}/hourly_stats"))

# IP behaviour
(ip_df
    .write.format("delta")
    .mode("overwrite")
    .save(f"{GOLD_PATH}/ip_behaviour"))

# Endpoint stats
(endpoint_df
    .write.format("delta")
    .mode("overwrite")
    .save(f"{GOLD_PATH}/endpoint_stats"))

print("All 3 Gold tables written successfully.")

All 3 Gold tables written successfully.


In [0]:
for table in ["hourly_stats", "ip_behaviour", "endpoint_stats"]:
    df = spark.read.format("delta").load(f"{GOLD_PATH}/{table}")
    print(f"{table}: {df.count():,} rows")

hourly_stats: 662 rows
ip_behaviour: 81,892 rows
endpoint_stats: 21,079 rows
